# ACCESSAI: First Proto

In [1]:
!pip install ultralytics

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ----------------------------------- ---- 1.3/1.5 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 6.5 MB/s  0:00:00
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ------------ --------------------------- 2.6/8.2 MB 15.1 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 22.1 MB/s  0:00:00
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 29.2 MB/s  0:00:00
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------- ----------------------------- 11.0/44.0 MB 52.7 MB/s eta 0:00:01
   --------------------- ------------------ 23.9/44.0 MB 55.9 MB/s eta 0:00:01
   ---------------------------------- ----- 37.5/44.0 MB 58.1 MB/s eta 0:00:01
   ---------------------------------------- 44.0/44.0 MB 53.8 MB/s  0:00:00
   ---------------------------------

ERROR: Could not install packages due to an OSError: [WinError 5] Acceso denegado: 'c:\\python310\\lib\\site-packages\\_distutils_hack\\override.py'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Python310\python.exe -m pip install --upgrade pip


In [4]:
!nvidia-smi

Failed to initialize NVML: GPU access blocked by the operating system


In [3]:
from ultralytics import YOLO

In [ ]:
DATASET_PATH = "dataset"  # Ajusta si es necesario

In [ ]:
# 1. Verificar la estructura de carpetas
print("Estructura del dataset:")
for split in ['train', 'valid', 'test']:
    split_path = os.path.join(DATASET_PATH, split)
    if os.path.exists(split_path):
        images_path = os.path.join(split_path, 'images')
        labels_path = os.path.join(split_path, 'labels')
        n_images = len(os.listdir(images_path)) if os.path.exists(images_path) else 0
        n_labels = len(os.listdir(labels_path)) if os.path.exists(labels_path) else 0
        print(f"  {split}: {n_images} imágenes, {n_labels} labels")
    else:
        print(f"  {split}: ❌ No existe")

# 2. Leer el data.yaml
yaml_path = os.path.join(DATASET_PATH, "data.yaml")
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("\nContenido del data.yaml:")
print(f"  nc: {data_config.get('nc')}")
print(f"  names: {data_config.get('names')}")
print(f"  train: {data_config.get('train')}")
print(f"  val: {data_config.get('val')}")
print(f"  test: {data_config.get('test')}")

#### IMPORTANTE: Habrá que reducir el número de clases y seleccionar aquellas que consideremos más importantes para la detección de barreras de accesibilidad urbana. También habrá que compensar el desbalance entre las clases seleccionadas si este existiera.

In [ ]:
# Descarga del modelo preentrenado y escaneado del conjunto de validación del dataset.
!yolo val model=yolo26n.pt data=dataset/data.yaml

#### mAP: mean Average Precision, métrica estándar para evaluar modelos de detección de objetos, que mide cuán bueno es el modelo detectando objetos y localizándolos correctamente. maP50 a 0.0333, MUY BAJO, el modelo intenta detectar las clases de nuestro dataset pero solo conoce las clases mediante las que ha sido entrenado

In [ ]:
model = YOLO("yolo26n.pt")

In [ ]:
# Entenamiento base del modelo, punto de partida:

results = model.train(
    data="dataset/data.yaml",
    epochs=50,          
    imgsz=640,          
    batch=16,
    device=0    # Activa la GPU
)

### Métricas del entrenamiento

Durante el entrenamiento, Ultralytics muestra una serie de métricas que reflejan el rendimiento del modelo. Estas son las más importantes:

#### Pérdidas (Losses)

| Métrica | ¿Qué mide? | ¿Qué esperar? |
| :--- | :--- | :--- |
| **box_loss** | Error en la localización de las bounding boxes (coordenadas). | Debe bajar con las épocas. |
| **cls_loss** | Error en la clasificación de cada objeto (¿es un coche? ¿una persona?). | Debe bajar con las épocas. |
| **l1_loss** | Pérdida auxiliar de regresión que ayuda a estabilizar el entrenamiento. | Debe bajar con las épocas. |

#### Métricas de evaluación (Box)

| Métrica | ¿Qué mide? | ¿Qué esperar? |
| :--- | :--- | :--- |
| **Precision (P)** | De todos los objetos que el modelo detecta, ¿cuántos son correctos? | Cuanto más alto, mejor. Cercano a 1.0 es ideal. |
| **Recall (R)** | De todos los objetos reales, ¿cuántos detecta el modelo? | Cuanto más alto, mejor. Cercano a 1.0 es ideal. |
| **mAP50** | Precisión media con un umbral de solapamiento (IoU) del 50%. | Cuanto más alto, mejor. >0.90 es excelente. |
| **mAP50-95** | Precisión media promediada sobre umbrales de IoU de 50% a 95%. | Cuanto más alto, mejor. >0.70 es muy bueno. |

#### Otras métricas

| Métrica | ¿Qué mide? |
| :--- | :--- |
| **Instances** | Número de objetos anotados en el batch actual. |
| **GPU_mem** | Memoria de la GPU utilizada durante el entrenamiento. |
| **Size** | Tamaño de las imágenes de entrada (ej. 640x640). |

#### ¿Qué es el IoU?

El **IoU** (Intersection over Union) mide el solapamiento entre la bounding box predicha y la real. Va de 0 (sin solapamiento) a 1 (solapamiento perfecto). Un IoU de 0.5 significa que la predicción cubre al menos el 50% del objeto real.

#### ¿Qué es el mAP?

El **mAP** (mean Average Precision) es la métrica estándar para evaluar modelos de detección de objetos. Combina Precision y Recall en un solo número. Un mAP alto significa que el modelo detecta los objetos correctamente y con buena localización.

Si el mAP es bueno (>0.90) → Documentar y pasar a la demo.

Si el mAP es bajo (<0.80) → Analizar qué clases fallan y considerar:

    Más épocas.

    Data augmentation específico.

    Ajuste de hiperparámetros con model.tune().

Si la demo falla mucho → Documentar la limitación de dominio y proponer fine-tuning con imágenes locales como trabajo futuro.

In [ ]:
model = YOLO("runs/detect/train-2/weights/best.pt")  # Cargamos el mejor modelo

In [ ]:
metrics = model.val(data="dataset/data.yaml", split="test")  # Evaluamos

In [ ]:
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("runs/detect/train-2/confusion_matrix_normalized.png")  # Matriz de confusión
plt.figure(figsize=(14, 12))
plt.imshow(img)
plt.axis('off')
plt.show()